# The Leakage Mirage — BMM NER Benchmark Audit

Quantifies how much train–test leakage inflates reported CRF and
BiLSTM-CRF scores on the Bhojpuri / Magahi / Maithili NER dataset
(Mundotiya et al.), and whether a size-matched control confirms the
drop is due to leakage removal rather than a smaller test set.

**Before running:** upload `bmmnerdataset.zip` to this Colab session's
file browser (already done, based on your screenshot) — this notebook
extracts it in the next cell.


In [ ]:
import sys
print(sys.version)

!pip --version

3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
pip 24.1.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)


In [ ]:
!pip uninstall -y sklearn-crfsuite

!pip install git+https://github.com/TeamHG-Memex/sklearn-crfsuite.git

Found existing installation: sklearn-crfsuite 0.5.0
Uninstalling sklearn-crfsuite-0.5.0:
  Successfully uninstalled sklearn-crfsuite-0.5.0
  Cloning https://github.com/TeamHG-Memex/sklearn-crfsuite.git to /tmp/pip-req-build-gprgaobf
  Running command git clone --filter=blob:none --quiet https://github.com/TeamHG-Memex/sklearn-crfsuite.git /tmp/pip-req-build-gprgaobf
  Resolved https://github.com/TeamHG-Memex/sklearn-crfsuite.git to commit 6efccac19a9e711ddea75a9196a44ad3c37ce260
  Preparing metadata (setup.py) ... done
  Using cached python_crfsuite-0.9.12-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (4.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 63.1 MB/s eta 0:00:00
  Created wheel for sklearn-crfsuite: filename=sklearn_crfsuite-0.3.6-py2.py3-none-any.whl size=10764 sha256=7e9c27428c06186640b28f3c5d3c09749524eb6e34ddec345aba32ad87141e01
  Stored in directory: /tmp/pip-ephem-wheel-cache-g62auyj6/wheels/e6/a4/1f/7916f0c86f40074b334fb8d82949a0ada

In [ ]:
import zipfile, os

ZIP_PATH = "bmmnerdataset.zip"   # adjust if you placed it elsewhere
EXTRACT_DIR = "bmm_data"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

FILES = {
    "Bhojpuri": ("Bhojpuri_ner.remove[1].train", "Bhojpuri_ner.remove[1].test"),
    "Magahi":   ("Magahi_ner[1].train",          "Magahi_ner[1].test"),
    "Maithili": ("Maithili_ner[1].train",        "Maithili_ner[1].test"),
}
FILES = {lang: (os.path.join(EXTRACT_DIR, tr), os.path.join(EXTRACT_DIR, te))
         for lang, (tr, te) in FILES.items()}
print(FILES)

{'Bhojpuri': ('bmm_data/Bhojpuri_ner.remove[1].train', 'bmm_data/Bhojpuri_ner.remove[1].test'), 'Magahi': ('bmm_data/Magahi_ner[1].train', 'bmm_data/Magahi_ner[1].test'), 'Maithili': ('bmm_data/Maithili_ner[1].train', 'bmm_data/Maithili_ner[1].test')}


## Step 1 — Parsing, degenerate-sentence filtering, leakage detection

`leakage_lib.py` logic, inlined so the notebook is self-contained.

Key design choices (see prior discussion for the reasoning):
- **Degenerate filtering**: sentences under 3 tokens with no entity
  are dropped before any leakage computation — otherwise punctuation-
  only fragments (e.g. a lone "।") inflate the leakage percentage
  without representing real memorizable content. This mattered a lot
  for Magahi specifically.
- **Leakage** = exact match + near-duplicate (token-Jaccard ≥ 0.9,
  length-gated to avoid short-sentence false positives).
- **Three test conditions, ONE shared trained model** (only the test
  set changes) so the comparison isolates leakage, not training
  differences:
  - `original` — full test set (leak + clean mixed, as originally
    released)
  - `clean_test_same_model` — leaked sentences removed from test
  - `size_matched_control` — same size as the clean test set, but
    randomly sampled from the FULL test set (so it keeps the natural
    leakage rate) — this is what isolates "did the score drop because
    of leakage, or just because N got smaller?"
- A fourth condition, `fully_clean`, also removes the leaking
  sentences from **train** and retrains, for the strictest version of
  the claim.


In [ ]:
"""
Leakage Mirage pipeline — core library.
Parsing, degenerate-sentence filtering, exact + near-duplicate leakage
detection, and matched control-split construction for the BMM
(Bhojpuri / Magahi / Maithili) NER dataset.
"""
import random
from collections import Counter

RANDOM_SEED = 42

# ---------------------------------------------------------------------
# 1. Parsing
# ---------------------------------------------------------------------
def read_conll(path):
    """Read a two-column (token \\t tag) CoNLL-style file into a list of
    sentences, where each sentence is a list of (token, tag) tuples."""
    sentences, cur = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip():
                if cur:
                    sentences.append(cur)
                    cur = []
                continue
            parts = line.split("\t")
            if len(parts) != 2:
                # tolerate stray malformed lines rather than crashing
                continue
            cur.append((parts[0], parts[1]))
    if cur:
        sentences.append(cur)
    return sentences


def sent_text(sent):
    return " ".join(tok for tok, _ in sent)


def n_entities(sent):
    return sum(1 for _, tag in sent if tag.startswith("B-"))


# ---------------------------------------------------------------------
# 2. Degenerate-sentence filtering
# ---------------------------------------------------------------------
def is_degenerate(sent, min_tokens=3):
    """A sentence is 'degenerate' if it is too short to carry real
    content (refrain fragments, stray punctuation lines, etc.) AND
    contains no entity. A short sentence that DOES contain a real
    entity is kept — we only want to drop noise, not information."""
    if len(sent) >= min_tokens:
        return False
    if n_entities(sent) > 0:
        return False
    return True


def clean_corpus(sentences, min_tokens=3):
    kept = [s for s in sentences if not is_degenerate(s, min_tokens)]
    dropped = len(sentences) - len(kept)
    return kept, dropped


# ---------------------------------------------------------------------
# 3. Leakage detection: exact + near-duplicate
# ---------------------------------------------------------------------
def jaccard(a, b):
    if not a or not b:
        return 0.0
    inter = len(a & b)
    union = len(a | b)
    return inter / union if union else 0.0


def find_leaked(train_sents, test_sents, near_dup_threshold=0.9,
                 near_dup_min_len=5, len_tolerance=0.2):
    """
    Returns (leaked_exact, leaked_near, non_leaked) — three lists of
    test sentences. near-duplicate matching is restricted to sentences
    with >= near_dup_min_len tokens (short-sentence Jaccard is noisy
    and produces false positives) and to train sentences of comparable
    length (for tractability and to avoid spurious matches).
    """
    train_exact = Counter(sent_text(s) for s in train_sents)
    train_tok_sets = [(len(s), set(tok for tok, _ in s)) for s in train_sents]

    leaked_exact, leaked_near, non_leaked = [], [], []
    for s in test_sents:
        text = sent_text(s)
        if text in train_exact:
            leaked_exact.append(s)
            continue
        if len(s) < near_dup_min_len:
            non_leaked.append(s)
            continue
        tset = set(tok for tok, _ in s)
        found = False
        for tlen, trs in train_tok_sets:
            if abs(tlen - len(s)) > max(2, len_tolerance * len(s)):
                continue
            if jaccard(tset, trs) >= near_dup_threshold:
                found = True
                break
        (leaked_near if found else non_leaked).append(s)
    return leaked_exact, leaked_near, non_leaked


# ---------------------------------------------------------------------
# 4. Matched control split
# ---------------------------------------------------------------------
def entity_bucket(sent):
    n = n_entities(sent)
    if n == 0:
        return "0"
    if n <= 2:
        return "1-2"
    return "3+"


def build_size_matched_control(full_test_sents, size, seed=RANDOM_SEED):
    """
    IMPORTANT: this samples from the FULL test pool (leaked + clean
    mixed, in their natural proportion) — NOT from the non-leaked pool
    alone. Sampling from the non-leaked pool would just reproduce the
    deduped test set itself once size == len(non-leaked pool), which
    is a no-op, not a control.

    The point of this sample is to isolate the "sample-size trap": if
    a same-N sample that still contains the natural leakage rate
    scores about the same as the full test set, the drop seen on the
    deduped test set is attributable to removing leakage, not to
    having fewer examples.
    """
    rng = random.Random(seed)
    pool = list(full_test_sents)
    rng.shuffle(pool)
    return pool[:size]


# ---------------------------------------------------------------------
# 5. Convenience: build all experimental conditions for a language
# ---------------------------------------------------------------------
def build_conditions(train_path, test_path, min_tokens=3,
                      near_dup_threshold=0.9, near_dup_min_len=5, seed=RANDOM_SEED):
    raw_train = read_conll(train_path)
    raw_test = read_conll(test_path)

    train, train_dropped = clean_corpus(raw_train, min_tokens)
    test, test_dropped = clean_corpus(raw_test, min_tokens)

    leaked_exact, leaked_near, non_leaked = find_leaked(
        train, test, near_dup_threshold, near_dup_min_len
    )
    leaked_all = leaked_exact + leaked_near
    deduped_test = non_leaked
    control_test = build_size_matched_control(test, len(deduped_test), seed=seed)

    # Sentences that are the source of leakage: drop them from train too,
    # for the strict "fully clean" condition (model never sees any
    # sentence, in either form, that also appears in test).
    leaked_texts = {sent_text(s) for s in leaked_all}
    train_leak_free = [s for s in train if sent_text(s) not in leaked_texts]

    conditions = {
        # (train, test) pairs. Conditions 1-3 share ONE trained model —
        # only the test set changes — see run_leakage_tax() below.
        "original": {"train": train, "test": test},
        "clean_test_same_model": {"train": train, "test": deduped_test},
        "size_matched_control": {"train": train, "test": control_test},
        # Condition 4 is the only one that also retrains on a different
        # training set (leak sentences removed from train as well).
        "fully_clean": {"train": train_leak_free, "test": deduped_test},
    }
    stats = {
        "train_sentences_raw": len(raw_train),
        "train_sentences_after_cleaning": len(train),
        "train_degenerate_dropped": train_dropped,
        "test_sentences_raw": len(raw_test),
        "test_sentences_after_cleaning": len(test),
        "test_degenerate_dropped": test_dropped,
        "leaked_exact": len(leaked_exact),
        "leaked_near_dup": len(leaked_near),
        "leaked_total": len(leaked_all),
        "leaked_pct_of_test": 100 * len(leaked_all) / len(test) if test else 0,
        "leaked_substantive_gt5tok_pct": (
            100 * sum(1 for s in leaked_all if len(s) > 5) / len(leaked_all)
            if leaked_all else 0
        ),
        "deduped_test_size": len(deduped_test),
        "control_test_size": len(control_test),
    }
    return conditions, stats


In [ ]:
import pandas as pd

all_conditions = {}
all_stats = {}
for lang, (trp, tep) in FILES.items():
    conditions, stats = build_conditions(trp, tep)
    all_conditions[lang] = conditions
    all_stats[lang] = stats

stats_df = pd.DataFrame(all_stats).T
stats_df

,train_sentences_raw,train_sentences_after_cleaning,train_degenerate_dropped,test_sentences_raw,test_sentences_after_cleaning,test_degenerate_dropped,leaked_exact,leaked_near_dup,leaked_total,leaked_pct_of_test,leaked_substantive_gt5tok_pct,deduped_test_size,control_test_size
Bhojpuri,11544.0,11123.0,421.0,4948.0,4758.0,190.0,2057.0,13.0,2070.0,43.505675,88.888889,2688.0,2688.0
Magahi,4255.0,3452.0,803.0,1065.0,855.0,210.0,370.0,49.0,419.0,49.005848,77.565632,436.0,436.0
Maithili,7848.0,7460.0,388.0,1966.0,1865.0,101.0,766.0,16.0,782.0,41.930295,92.327366,1083.0,1083.0


## Step 2 — CRF baseline (locked hyperparameters)

Same `sklearn-crfsuite` settings for every language and every
condition — only the data changes. Runs on CPU; a few minutes total
for all three languages.


In [ ]:
"""
CRF baseline — sklearn-crfsuite, locked hyperparameters across all
experimental conditions. Only the data changes between runs.
"""
import sklearn_crfsuite
from seqeval.metrics import f1_score, classification_report

# Locked hyperparameters — identical for every condition/language.
CRF_PARAMS = dict(
    algorithm="lbfgs",
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True,
)


def word2features(sent_tokens, i):
    word = sent_tokens[i]
    features = {
        "bias": 1.0,
        "word.lower()": word.lower(),
        "word[-3:]": word[-3:],
        "word[-2:]": word[-2:],
        "word[:3]": word[:3],
        "word.isdigit()": word.isdigit(),
        "word.len": len(word),
        "BOS": i == 0,
        "EOS": i == len(sent_tokens) - 1,
    }
    if i > 0:
        prev = sent_tokens[i - 1]
        features.update({
            "-1:word.lower()": prev.lower(),
            "-1:word[-3:]": prev[-3:],
        })
    if i < len(sent_tokens) - 1:
        nxt = sent_tokens[i + 1]
        features.update({
            "+1:word.lower()": nxt.lower(),
            "+1:word[-3:]": nxt[-3:],
        })
    return features


def sent2features(sent):
    tokens = [tok for tok, _ in sent]
    return [word2features(tokens, i) for i in range(len(tokens))]


def sent2labels(sent):
    return [tag for _, tag in sent]


def train_crf(train_sents, params=None):
    params = params or CRF_PARAMS
    X_train = [sent2features(s) for s in train_sents]
    y_train = [sent2labels(s) for s in train_sents]
    crf = sklearn_crfsuite.CRF(**params)
    crf.fit(X_train, y_train)
    return crf


def evaluate_crf(crf, test_sents):
    X_test = [sent2features(s) for s in test_sents]
    y_test = [sent2labels(s) for s in test_sents]
    y_pred = crf.predict(X_test)
    micro_f1 = f1_score(y_test, y_pred, average="micro")
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    report = classification_report(y_test, y_pred, digits=3, zero_division=0)
    return {"micro_f1": micro_f1, "macro_f1": macro_f1, "report": report}


def run_crf_leakage_tax(conditions):
    """
    Trains the CRF ONCE on the original (leaky) train set and evaluates
    it on three test sets (original/clean_test_same_model/size_matched_
    control) — this is the correct comparison, since composition of the
    TEST set is the variable under study, not the model. Separately
    trains a second model for 'fully_clean' (different train set too).
    """
    results = {}
    shared_model = train_crf(conditions["original"]["train"])
    for cond in ("original", "clean_test_same_model", "size_matched_control"):
        results[cond] = evaluate_crf(shared_model, conditions[cond]["test"])

    clean_model = train_crf(conditions["fully_clean"]["train"])
    results["fully_clean"] = evaluate_crf(clean_model, conditions["fully_clean"]["test"])
    return results


ModuleNotFoundError: No module named 'seqeval'

In [ ]:
crf_results = {}
for lang, conditions in all_conditions.items():
    print(f"Training CRF — {lang} ...")
    crf_results[lang] = run_crf_leakage_tax(conditions)

crf_rows = []
for lang, res in crf_results.items():
    for cond, metrics in res.items():
        crf_rows.append({
            "language": lang, "model": "CRF", "condition": cond,
            "test_n": len(all_conditions[lang][cond]["test"]),
            "micro_f1": metrics["micro_f1"], "macro_f1": metrics["macro_f1"],
        })
crf_df = pd.DataFrame(crf_rows)
crf_df

## Step 3 — BiLSTM-CRF baseline (locked hyperparameters)

**Note on this section:** the CRF pipeline above was fully executed
against the real dataset while this notebook was built, and the
results table above is real, not illustrative. This BiLSTM-CRF section
was carefully written and reviewed but could **not** be execution-
tested in the environment that built this notebook (no GPU/torch
available there) — run the smoke test cell below first (2 epochs) to
confirm it runs cleanly on your Colab GPU before committing to the
full 15-epoch runs across all languages and conditions.


In [ ]:
"""
BiLSTM-CRF baseline — PyTorch + pytorch-crf, locked hyperparameters
across all experimental conditions. Vocabulary/label maps are built
fresh per training set (since the train set differs across
conditions), but every architectural and optimization hyperparameter
is fixed, per the "hyperparameter locking" requirement.
"""
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchcrf import CRF
from seqeval.metrics import f1_score

# ---------------------------------------------------------------------
# Locked hyperparameters — identical for every condition/language.
# ---------------------------------------------------------------------
SEED = 42
EMBED_DIM = 100
HIDDEN_DIM = 128
BATCH_SIZE = 16
EPOCHS = 15
LR = 1e-3
PAD = "<PAD>"
UNK = "<UNK>"


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def build_vocab(train_sents):
    words = {PAD, UNK}
    tags = {PAD}
    for s in train_sents:
        for tok, tag in s:
            words.add(tok)
            tags.add(tag)
    word2idx = {w: i for i, w in enumerate(sorted(words))}
    tag2idx = {t: i for i, t in enumerate(sorted(tags))}
    return word2idx, tag2idx


class NERDataset(Dataset):
    def __init__(self, sentences, word2idx, tag2idx):
        self.sentences = sentences
        self.word2idx = word2idx
        self.tag2idx = tag2idx

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sent = self.sentences[idx]
        word_ids = [self.word2idx.get(tok, self.word2idx[UNK]) for tok, _ in sent]
        tag_ids = [self.tag2idx.get(tag, self.tag2idx[PAD]) for _, tag in sent]
        return torch.tensor(word_ids), torch.tensor(tag_ids)


def collate(batch, pad_word_id, pad_tag_id):
    words, tags = zip(*batch)
    lengths = torch.tensor([len(w) for w in words])
    max_len = lengths.max().item()
    word_padded = torch.full((len(batch), max_len), pad_word_id, dtype=torch.long)
    tag_padded = torch.full((len(batch), max_len), pad_tag_id, dtype=torch.long)
    mask = torch.zeros((len(batch), max_len), dtype=torch.bool)
    for i, (w, t) in enumerate(zip(words, tags)):
        word_padded[i, : len(w)] = w
        tag_padded[i, : len(t)] = t
        mask[i, : len(w)] = True
    return word_padded, tag_padded, mask


class BiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, tagset_size, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM,
                 pad_word_id=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_word_id)
        self.lstm = nn.LSTM(embed_dim, hidden_dim // 2, num_layers=1,
                             bidirectional=True, batch_first=True)
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)
        self.crf = CRF(tagset_size, batch_first=True)

    def _emissions(self, words):
        embeds = self.embedding(words)
        lstm_out, _ = self.lstm(embeds)
        return self.hidden2tag(lstm_out)

    def loss(self, words, tags, mask):
        emissions = self._emissions(words)
        return -self.crf(emissions, tags, mask=mask, reduction="mean")

    def predict(self, words, mask):
        emissions = self._emissions(words)
        return self.crf.decode(emissions, mask=mask)


def train_bilstm_crf(train_sents, device="cpu", epochs=EPOCHS, seed=SEED, verbose=False):
    set_seed(seed)
    word2idx, tag2idx = build_vocab(train_sents)
    idx2tag = {i: t for t, i in tag2idx.items()}
    pad_word_id, pad_tag_id = word2idx[PAD], tag2idx[PAD]

    ds = NERDataset(train_sents, word2idx, tag2idx)
    loader = DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=lambda b: collate(b, pad_word_id, pad_tag_id),
    )

    model = BiLSTM_CRF(len(word2idx), len(tag2idx), pad_word_id=pad_word_id).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for words, tags, mask in loader:
            words, tags, mask = words.to(device), tags.to(device), mask.to(device)
            optimizer.zero_grad()
            loss = model.loss(words, tags, mask)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if verbose:
            print(f"  epoch {epoch+1}/{epochs}  loss={total_loss/len(loader):.4f}")

    return model, word2idx, tag2idx, idx2tag


def evaluate_bilstm_crf(model, word2idx, tag2idx, idx2tag, test_sents, device="cpu"):
    pad_word_id, pad_tag_id = word2idx[PAD], tag2idx[PAD]
    ds = NERDataset(test_sents, word2idx, tag2idx)
    loader = DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=lambda b: collate(b, pad_word_id, pad_tag_id),
    )
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for words, tags, mask in loader:
            words, tags, mask = words.to(device), tags.to(device), mask.to(device)
            preds = model.predict(words, mask)
            for i, seq in enumerate(preds):
                length = mask[i].sum().item()
                true_tags = [idx2tag[t.item()] for t in tags[i][:length]]
                pred_tags = [idx2tag[t] for t in seq]
                y_true.append(true_tags)
                y_pred.append(pred_tags)
    return {
        "micro_f1": f1_score(y_true, y_pred, average="micro"),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }


def run_bilstm_leakage_tax(conditions, device="cpu"):
    """Mirrors run_crf_leakage_tax: one shared model for the three
    test-set-only conditions, a separate model for fully_clean."""
    results = {}
    model, w2i, t2i, i2t = train_bilstm_crf(conditions["original"]["train"], device=device)
    for cond in ("original", "clean_test_same_model", "size_matched_control"):
        results[cond] = evaluate_bilstm_crf(model, w2i, t2i, i2t, conditions[cond]["test"], device)

    model2, w2i2, t2i2, i2t2 = train_bilstm_crf(conditions["fully_clean"]["train"], device=device)
    results["fully_clean"] = evaluate_bilstm_crf(
        model2, w2i2, t2i2, i2t2, conditions["fully_clean"]["test"], device
    )
    return results


In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# --- Smoke test: 2 epochs on the smallest language before trusting the full run ---
smoke_conditions = all_conditions["Magahi"]
smoke_model, w2i, t2i, i2t = train_bilstm_crf(smoke_conditions["original"]["train"],
                                               device=device, epochs=2, verbose=True)
smoke_result = evaluate_bilstm_crf(smoke_model, w2i, t2i, i2t,
                                    smoke_conditions["original"]["test"], device=device)
print("Smoke test result (2 epochs, Magahi, original):", smoke_result)

In [ ]:
# Full run — all languages, all conditions. EPOCHS is set in bilstm_crf_model.py (default 15).
bilstm_results = {}
for lang, conditions in all_conditions.items():
    print(f"Training BiLSTM-CRF — {lang} ...")
    bilstm_results[lang] = run_bilstm_leakage_tax(conditions, device=device)

bilstm_rows = []
for lang, res in bilstm_results.items():
    for cond, metrics in res.items():
        bilstm_rows.append({
            "language": lang, "model": "BiLSTM-CRF", "condition": cond,
            "test_n": len(all_conditions[lang][cond]["test"]),
            "micro_f1": metrics["micro_f1"], "macro_f1": metrics["macro_f1"],
        })
bilstm_df = pd.DataFrame(bilstm_rows)
bilstm_df

## Step 4 — Combined "Leakage Tax" table

The headline number for the paper: `original` vs `clean_test_same_model`
F1, with `size_matched_control` reported alongside as the check that
rules out "smaller test set" as the explanation.


In [ ]:
results_df = pd.concat([crf_df, bilstm_df], ignore_index=True)
results_df.to_csv("leakage_tax_results.csv", index=False)

pivot = results_df.pivot_table(index=["language", "model"], columns="condition",
                                values=["micro_f1", "macro_f1"])
pivot["leakage_tax_micro_f1"] = (
    pivot[("micro_f1", "original")] - pivot[("micro_f1", "clean_test_same_model")]
)
pivot["leakage_tax_macro_f1"] = (
    pivot[("macro_f1", "original")] - pivot[("macro_f1", "clean_test_same_model")]
)
pivot

In [ ]:
stats_df.to_csv("leakage_audit_stats.csv")
print("Saved: leakage_tax_results.csv, leakage_audit_stats.csv")